In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import geopandas as gpd
from PIL import Image
import glob
from pyprojroot import here

In [2]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results"

In [3]:
gdf = gpd.read_file(RAW_DATA_DIR/'geographic/municipalities/00mun.shp')
df = pd.read_csv(INTERIM_DATA_DIR/"crime_count_2015-2025.csv", encoding='latin-1')

In [4]:
df["CVEGEO"] = df["CVEGEO"].astype(str).str.zfill(5)
gdf["CVEGEO"] = gdf["CVEGEO"].astype(str).str.zfill(5)

In [5]:
pivot = df.pivot_table(index="CVEGEO", columns="year",
                        values="crime_count", aggfunc="sum")

years = df['year'].unique().tolist()

In [6]:
merged = gdf[["CVEGEO", "geometry"]].merge(pivot, on="CVEGEO", how="left")

# Optional sanity check: municipalities with no match
unmatched = merged[merged[years].isna().all(axis=1)]
if len(unmatched) > 0:
    print(f"Warning: {len(unmatched)} municipalities have no crime data match")

In [7]:
vmin = np.log1p(merged[years].min().min())
vmax = np.log1p(merged[years].max().max())
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = plt.cm.YlOrRd

In [8]:
for year in years:
    fig, ax = plt.subplots(figsize=(8, 8))

    merged["_log_val"] = np.log1p(merged[year])
    merged.plot(
        column="_log_val",
        cmap=cmap,
        norm=norm,
        linewidth=0.05,
        edgecolor="grey",
        ax=ax,
        missing_kwds={"color": "lightgrey", "label": "No data"},
    )

    ax.set_title(f"Municipal Crime Count — {year}", fontsize=14)
    ax.axis("off")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
    cbar.set_label("log(1 + crime_count)")

    plt.tight_layout()

    plt.savefig(RESULTS_DATA_DIR/f"raw_descriptive_analysis/{year}_crime_heatmap.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

### Crime counts 2015-2025 .gif

In [9]:
out_dir = RESULTS_DATA_DIR / "raw_descriptive_analysis"
gif_path = RESULTS_DATA_DIR / "raw_descriptive_analysis" / "crime_heatmap_animation.gif"

frame_paths = sorted(
    glob.glob(f"{out_dir}/*_crime_heatmap.png"),
    key=lambda p: int(p.split("_crime_heatmap.png")[0].split("/")[-1].split("\\")[-1])
)

frames = [Image.open(p).convert("RGB") for p in frame_paths]

frames[0].save(
    gif_path,
    save_all=True,
    append_images=frames[1:],
    duration=1000,      # ms per frame
    loop=0,             # loop forever
)

print(f"Saved GIF with {len(frames)} frames to {gif_path}")

Saved GIF with 11 frames to C:\Users\franc\OneDrive\Documents\Repositories\organized-crime-prediction\results\raw_descriptive_analysis\crime_heatmap_animation.gif
